# Bank Customer Churn Prediction

**End-to-end machine learning classification project**

This project explores customer churn in a banking dataset and builds a machine learning pipeline to predict whether a customer is likely to leave the bank.

**Core tools:** Python · Pandas · NumPy · Scikit-learn · Matplotlib · Seaborn

> **Portfolio note:** The dataset used in the original notebook is not included in this repository. Place the CSV file in `data/` and update `DATA_PATH` below before rerunning the notebook.


## 1. Problem Statement

Customer churn is a classification problem in which the model predicts whether a customer will exit the bank.

The target variable is **`Exited`**:

- `0` — No churn
- `1` — Churn

The objective is to compare several classification algorithms and select a model based on cross-validated ROC-AUC, followed by evaluation on a held-out test set.


In [ ]:
from pathlib import Path
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVC

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
sns.set_theme(style="whitegrid")

DATA_PATH = Path("../data/CUSTOMER CHURN RECORDS.csv")


## 2. Load and Inspect the Dataset


In [ ]:
# Update DATA_PATH if your CSV has a different filename.
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. "
        "Place the CSV in the repository's data/ folder."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.describe().T


In [ ]:
df.isnull().sum()


### Dataset observations

The original analysis found:

- **10,000 observations**
- **13 columns**
- No missing values in the original dataset
- The target contained **7,962 non-churned customers** and **2,038 churned customers** (20.38% churn)

The original notebook therefore represents a moderately imbalanced binary classification problem.


## 3. Exploratory Data Analysis


In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="Exited")
plt.title("Customer Churn Distribution")
plt.xlabel("Exited")
plt.ylabel("Number of Customers")
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Geography", hue="Exited")
plt.title("Churn Distribution by Geography")
plt.xlabel("Geography")
plt.ylabel("Number of Customers")
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Exited", y="Age")
plt.title("Age Distribution by Churn Status")
plt.xlabel("Exited")
plt.ylabel("Age")
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
sns.scatterplot(data=df, x="NumOfProducts", y="Balance", hue="Exited", alpha=0.7)
plt.title("Balance vs Number of Products by Churn")
plt.show()


In [ ]:
churn_rate = df.groupby("NumOfProducts")["Exited"].mean().reset_index()

plt.figure(figsize=(8, 5))
sns.barplot(data=churn_rate, x="NumOfProducts", y="Exited")
plt.title("Churn Rate vs Number of Products")
plt.xlabel("Number of Products")
plt.ylabel("Churn Rate")
plt.show()


## 4. Feature Engineering


In [ ]:
df_fe = df.copy()

# Balance per product
df_fe["Balance per product"] = (
    df_fe["Balance"] / df_fe["NumOfProducts"].replace(0, np.nan)
).fillna(0)

# Salary-to-balance ratio
df_fe["Salary to Balance ratio"] = (
    df_fe["EstimatedSalary"] / df_fe["Balance"].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan)

df_fe["Salary to Balance ratio"] = (
    df_fe["Salary to Balance ratio"]
    .fillna(df_fe["Salary to Balance ratio"].median())
)

# Age groups
df_fe["AgeGroup"] = pd.cut(
    df_fe["Age"],
    bins=[0, 20, 30, 40, 50, 60, 100],
    labels=["<20", "20-29", "30-39", "40-49", "50-59", "60+"]
)

df_fe[[
    "Balance per product",
    "Salary to Balance ratio",
    "AgeGroup"
]].head()


Three derived features were created in the original project:

- **Balance per product**
- **Salary-to-balance ratio**
- **AgeGroup**

These features were intended to provide additional information for the classification models.


## 5. Prepare Features and Target


In [ ]:
target = "Exited"

# Identifier columns are not used as predictive features.
drop_col = ["Customer_Id"]  # Kept aligned with the original notebook

features = [c for c in df_fe.columns if c not in [target] + drop_col]

X = df_fe[features]
y = df_fe[target]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)


## 6. Preprocessing Pipeline


In [ ]:
numeric_features = [
    "CreditScore",
    "Age",
    "Tenure",
    "Balance",
    "NumOfProducts",
    "EstimatedSalary",
    "Balance per product",
    "Salary to Balance ratio"
]

categorical_features = [
    "Geography",
    "Gender",
    "IsActiveMember",
    "AgeGroup"
]

df_fe[categorical_features] = df_fe[categorical_features].astype("object")

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])


## 7. Train/Test Split


In [ ]:
X = df_fe[features]
y = df_fe[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)
print("Train churn proportion:", y_train.mean())
print("Test churn proportion:", y_test.mean())


## 8. Model Comparison with 5-Fold Stratified Cross-Validation


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    scores = cross_val_score(
        pipe, X_train, y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    results[name] = scores

    print(
        f"{name}: mean ROC-AUC={scores.mean():.4f}, "
        f"std={scores.std():.4f}"
    )


### Cross-validation results from the original analysis

| Model | Mean ROC-AUC | Std. Dev. |
|---|---:|---:|
| Logistic Regression | 0.7813 | 0.0098 |
| Random Forest | 0.8501 | 0.0096 |
| Gradient Boosting | **0.8611** | 0.0078 |
| SVM | 0.8224 | 0.0166 |
| KNN | 0.7821 | 0.0154 |

Gradient Boosting achieved the highest mean cross-validated ROC-AUC in the original experiment.


In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=[results[m] for m in results])
plt.xticks(range(len(results)), list(results.keys()), rotation=30, ha="right")
plt.ylabel("ROC-AUC")
plt.title("Model Comparison — 5-Fold Cross-Validated ROC-AUC")
plt.tight_layout()
plt.show()


## 9. Train and Evaluate the Selected Model


In [ ]:
best_name = max(results, key=lambda k: results[k].mean())
best_model = models[best_name]

best_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", best_model)
])

best_pipeline.fit(X_train, y_train)

y_pred = best_pipeline.predict(X_test)
y_proba = best_pipeline.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1 Score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_proba)
}

print("Selected model:", best_name)
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")


### Held-out test results

The original notebook reported the following results for Gradient Boosting on the 20% test set:

| Metric | Score |
|---|---:|
| Accuracy | **0.8660** |
| Precision | **0.7593** |
| Recall | **0.5025** |
| F1 Score | **0.6047** |
| ROC-AUC | **0.8775** |

These results should be interpreted together rather than relying on accuracy alone, particularly because churn is the minority class.


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=["No Churn", "Churn"],
    yticklabels=["No Churn", "Churn"]
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))


## 10. Save the Trained Pipeline


In [ ]:
MODEL_PATH = Path("../models/best_churn_pipeline.pkl")
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(best_pipeline, MODEL_PATH)
print(f"Saved pipeline to: {MODEL_PATH}")


## 11. Example Prediction

The original project also demonstrated how the trained pipeline can be used to generate a churn prediction and probability for a new customer.

For a production-style repository, the prediction logic should live in a reusable Python script rather than only inside the notebook.


In [ ]:
# Example customer
sample = {
    "RowNumber": 10001,
    "Surname": "Bien",
    "CustomerId": 15647322,
    "CreditScore": 890,
    "Age": 35,
    "Geography": "Spain",
    "Gender": "Male",
    "Balance": 0.00,
    "Tenure": 2,
    "EstimatedSalary": 101350.90,
    "IsActiveMember": 1,
    "NumOfProducts": 1
}

sample_df = pd.DataFrame([sample])

sample_df["Balance per product"] = (
    sample_df["Balance"] /
    sample_df["NumOfProducts"].replace(0, np.nan)
).fillna(0)

sample_df["Salary to Balance ratio"] = (
    sample_df["EstimatedSalary"] /
    sample_df["Balance"].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan)

sample_df["Salary to Balance ratio"] = (
    sample_df["Salary to Balance ratio"]
    .fillna(df_fe["Salary to Balance ratio"].median())
)

sample_df["AgeGroup"] = pd.cut(
    sample_df["Age"],
    bins=[0, 20, 30, 40, 50, 60, 100],
    labels=["<20", "20-29", "30-39", "40-49", "50-59", "60+"]
)

prediction = best_pipeline.predict(sample_df[features])
probability = best_pipeline.predict_proba(sample_df[features])[:, 1]

print("Predicted Churn:", int(prediction[0]))
print("Churn Probability:", round(float(probability[0]), 4))


## 12. Key Takeaways

- The project uses an end-to-end Scikit-learn pipeline for preprocessing and classification.
- Five classification approaches were compared using stratified 5-fold cross-validation.
- Gradient Boosting produced the highest mean cross-validated ROC-AUC in this experiment (**0.8611**).
- On the held-out test set, the model achieved a ROC-AUC of **0.8775** and accuracy of **0.8660**.
- The project demonstrates practical skills in EDA, feature engineering, preprocessing, model comparison, evaluation, and model persistence.
